# 🧠 Foundational PyTorch Pipeline: First-Principles Architecture

This notebook serves as the master blueprint for building neural networks from absolute scratch. It is designed with a **first-principles approach**, focusing on memory efficiency, hardware optimization (crucial for standalone offline hardware), and robust Kaggle-grade pipelines.

### Core Concepts Mastered Here:
* **Dynamic Hardware Routing:** Teleporting data safely between CPU (Island A) and GPU (Island B).
* **The Data Pipeline:** Understanding the `Dataset` (Warehouse) and `DataLoader` (Conveyor Belt).
* **Blueprint vs. Assembly Line:** Proper instantiation of `nn.Module` classes and the sacred `__init__` dunder.
* **The Windows Trap:** Managing `num_workers` and multiprocessing in Jupyter.
* **The Blindfolded Hiker:** Balancing Batch Size and Learning Rate for Tabular Data.

In [ ]:
import numpy as np
import pandas as pd
import torch as t
import torch.nn as nn   

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

print(f"PyTorch Version: {t.__version__}")

---
## 1. Data Loading & The "Fail Fast" Guard
Before feeding data into a neural network, we must ensure there are no poisoned variables (NaNs). A single `NaN` will destroy the matrix multiplication. We drop ghost columns and securely map target variables.

In [ ]:
# Load Dataset
df = pd.read_csv(r'C:\Users\Banwa\Desktop\IWT\CODING\Aresnal\ML Practice\Breast_Cancer_Dataset.csv')

# 1. The Poison Guard: Drop the ghost column if it exists
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])

# 2. Safely map text targets to binary integers
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

print("Data loaded and sanitized. Shape:", df.shape)

---
## 2. Preprocessing & The Tensor Bridge
Neural Networks cannot read Pandas DataFrames. We must convert them into PyTorch Tensors (Lego bricks). 

**Hardware Note:** We use `t.as_tensor()` to cleanly adapt NumPy arrays into `float32` tensors. This takes up exactly half the memory of `float64`, saving battery and VRAM during execution.

In [ ]:
# Split Features & Targets
X_train, X_test, Y_train, Y_test = train_test_split(
    df.iloc[:, 2:], # Skipping 'id' and 'diagnosis'
    df['diagnosis'],
    test_size=0.2,
    random_state=42 
)

# Standardization (Squishing the mountain so the AI doesn't trip)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# The Tensor Bridge (Converting raw arrays into PyTorch memory)
X_train_tensor = t.as_tensor(X_train, dtype=t.float32)
X_test_tensor = t.as_tensor(X_test, dtype=t.float32)
Y_train_tensor = t.as_tensor(np.array(Y_train), dtype=t.float32).view(-1, 1)
Y_test_tensor = t.as_tensor(np.array(Y_test), dtype=t.float32).view(-1, 1)

print(f"X Train Shape: {X_train_tensor.shape} | Y Train Shape: {Y_train_tensor.shape}")

---
## 3. The Data Pipeline (Warehouse & Conveyor Belt)

**1. `Dataset` (The Warehouse):** This class doesn't load all data at once. It acts as a set of instructions on how to fetch *one single item* from the hard drive. Crucial for massive image datasets to prevent RAM overflow.

**2. `DataLoader` (The Conveyor Belt):** This machine uses a `Sampler` (the manager choosing which indices to pick) and a `Collate Function` (the machine stacking them into a batch) to feed the GPU.

🚨 **The Windows Mutli-Processing Trap:** Because Windows uses `spawn` instead of `fork`, passing `num_workers > 0` inside a "floating" Jupyter cell will cause an infinite loop of cloning and crash the system. For local tabular data, we lock `num_workers=0`.

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        # Grabs exactly one patient's features and their diagnosis
        return self.features[index], self.labels[index]

# Instantiate the Warehouses
Traindataset = CustomDataset(X_train_tensor, Y_train_tensor)
Testdataset = CustomDataset(X_test_tensor, Y_test_tensor)

# Start the Conveyor Belts
TrainLoader = DataLoader(Traindataset, batch_size=128, shuffle=True, pin_memory=True, num_workers=0)
TestLoader = DataLoader(Testdataset, batch_size=128, shuffle=False, pin_memory=True, num_workers=0)

print(f"Train Dataset Length: {len(Traindataset)}")

---
## 4. The Neural Architecture (`nn.Module`)

PyTorch models are Object-Oriented Factories.
* **`__init__` (The Blueprint):** This is the workshop. We pass configurations (like `num_features`) here to build the physical math machines (Layers). **Never pass raw data here.**
* **`forward` (The Assembly Line):** This is where we pass the raw data (tensors) through the machines we built in the workshop.

In [ ]:
class My_NN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        # Building the components in the workshop
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        # Running data through the assembly line
        x = self.linear(features)
        x = self.sigmoid(x)
        return x 

---
## 5. The Training Loop

### Core Concepts:
1.  **Just-In-Time Teleportation:** Instead of sending 50GB of data to the GPU (which crashes VRAM), we only `.to(device)` the tiny batch of 128 items right before the math happens.
2.  **The Judge (Criterion):** We build `nn.BCELoss` *outside* the loop so we don't accidentally burn memory rebuilding the machine 10,000 times.
3.  **The Blindfolded Hiker:** Because we are using Mini-Batches, the model can't see the whole mountain. We must lower the learning rate (`lr=0.1` -> `lr=0.01`) so it takes careful steps and doesn't bounce around the minimum.

In [ ]:
# Hardware Router: Teleport the entire factory to the active hardware
device = t.device("cuda" if t.cuda.is_available() else "cpu")
print(f"Executing Math on: [{device}]")

lr     = 0.01
epochs = 200
model  = My_NN(X_train_tensor.shape[1]).to(device)  

# Instantiate the Judge (Outside the loop!)
criterion = nn.BCELoss()
optim     = t.optim.Adam(model.parameters(), lr=lr) # Adam is superior for Tabular data

for epoch in range(epochs):
    for batch_features, batch_labels in TrainLoader:
        # 1. Just-In-Time Teleportation (Island A -> Island B)
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Wipe the gradient chalkboard
        optim.zero_grad()

        # 2. Forward Pass
        y_pred = model(batch_features)
        
        # Make absolute sure we compare against batch_labels, not the whole dataset!
        predictions_class = (y_pred >= 0.5).float()
        accuracy = (predictions_class == batch_labels).float().mean() * 100

        # 3. Loss & Backward Pass
        loss = criterion(y_pred, batch_labels)
        loss.backward()
        optim.step()

    # Print progress at the end of the epoch
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d}/{epochs} | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.2f}%")

print("\nTraining complete.")

---
## 6. Evaluation & Freezing
We transition the model from `train()` mode to `eval()` mode. This turns off dynamic training features (like Dropout) and guarantees deterministic predictions. We also wrap the pass in `torch.no_grad()` to turn off the calculus tape-recorder, vastly increasing inference speed.

In [ ]:
model.eval() # Lock the brain

for batch_features, batch_labels in TestLoader:
    batch_labels = batch_labels.to(device)
    batch_features = batch_features.to(device)
    
    with t.no_grad(): # Turn off gradient tracking for maximum hardware efficiency
        test_preds = model(batch_features)
        test_preds_class = (test_preds >= 0.5).float()
        test_acc = (test_preds_class == batch_labels).float().mean() * 100
        print(f"Final Test Accuracy: {test_acc.item():.2f}%")
        
model.train() # Unlock the brain

# Print the architecture summary
summary(model, input_size=(1, 30))